上一章我们把一堆人类的句子，变成了（Batch， Seq_len, Dim）d的张量，现在我们把张量喂给Transformer，我们要创建它的大脑用于消化吸收这些信息，这个大脑就是多头注意力机制。

我们先来了解什么是注意力：
注意力机制的核心公式：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

那QKV又是啥呢？如果是有技术背景的同学可能会比较清楚，用查百科大全来类比：
Q Query 查询：你心中的问题---听说有种水果叫香蕉，让我查查？---“我要找什么”
K Key 键： 百科大全中这个词条的关键字---香蕉---“这个元素怎么表述”
V Value 值： 百科大全中词条下的具体内容--- 黄色，热带水果，弯的等等---“这个元素在实际世界中的信息”


计算过程：
Q @ K^T 用“问题”和所有的“词条”做点击，算出每个词条和我的问题有多相关（相关性分数）
/ √d_k 因为向量维度 d_k 越大，点积结果的绝对值越大，会导致 softmax 输出接近 one-shot（梯度消失），除以 √d_k 能把分数拉回。
| 输入 | softmax 输出 | 特征 |
|------|-------------|------|
| `[1, 2, 3]` | `[0.09, 0.24, 0.67]` | 分布比较均匀，梯度正常 |
| `[10, 20, 30]` | `[0.00, 0.00, 1.00]` | 接近 one-hot，梯度几乎为 0 |
softmax 把分数归一化为概率分布（加起来等于1）
```
输入：[2.0, 1.0, 0.1]
       ↓ softmax
输出：[0.659, 0.242, 0.099]   ← 全部为正，加起来 = 1
```
@V 用概率对所有“内容”做加权求和，得到最终输出。


多头 Multi - Head 的直觉
把QKV拆分成多个“头”，比如8个，相当于让模型有8个不同的大脑，每个脑子学习不同的方向，有的学习语法，有的学习语义，有的学习指代关系，最后把所有头拼接到一起，综合全部的专家建议。


In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除"

        # 这里注意一个细节，必须是 // ，/ 下面代码会报错，因为d_k必须是整数，
        self.d_k = d_model // num_heads  # 每个头的维度，比如 512 / 8 = 64
        self.n_heads = num_heads
        self.d_model = d_model

        #定义 qkv的权重矩阵

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)

        self.fc = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        # 线性比那还 + 拆分头
        # 变换前: (Batch, Len, D_model)
        # 变换后: (Batch, Len, n_head, d_k)
        # Permute后: (Batch, n_head, Len, d_k) -> 让 n_head 在前面，方便并行计算
        Q = self.w_q(q).view(batch_size, -1, self.n_heads, self.d_k).permute(0, 2, 1, 3)
        K = self.w_k(k).view(batch_size, -1, self.n_heads, self.d_k).permute(0, 2, 1, 3)
        V = self.w_v(v).view(batch_size, -1, self.n_heads, self.d_k).permute(0, 2, 1, 3)

        # 计算点积注意力，这里对K做转置，只交换最后两维，
        # 因为Q是(Batch, n_head, Len, d_k)，K^T是(Batch, n_head, d_k, Len)，矩阵乘法只计算最后两维度，其他维度都当做batch处理
        # 所以QK^T的形状是(Batch, n_head, Len, Len)，scores代表着第batch个样本，第h个头中， Q的第i个位置对K的第j个位置的注意力分数。
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # 应用mask
        if mask is not None:
            # mask的形状是和上面的scores一致
            # masked_fill 如果mask = 0，就把scores 对应位置改为 -1e9（负无穷），这样softmax之后就会变为0
            scores = scores.masked_fill(mask == 0, -1e9)

        attn = F.softmax(scores, dim=-1)

        # 对所有内容进行加权
        context = torch.matmul(attn, V)

        # 拼接多个头
        context = context.permute(0, 2, 1, 3).contiguous().view(batch_size, -1, self.d_model)
        output = self.fc(context)
        return output, attn  #这里返回attn是为了可视化


# --- 测试一下 ---
d_model = 512
num_heads = 8
mha = MultiHeadAttention(d_model, num_heads)

x = torch.randn(2, 10, 512)  #(batch, seq_Len, dim)
out, attm_map = mha(x, x, x, mask=None)
print(f"MultiHead Input: {x.shape}")
print(f"MultiHead Output: {out.shape}")
print(f"MultiHead Attention Output: {attm_map.shape}")


MultiHead Input: torch.Size([2, 10, 512])
MultiHead Output: torch.Size([2, 10, 512])
MultiHead Attention Output: torch.Size([2, 8, 10, 10])


In [12]:
# 再来一层前馈神经网络，
class FFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(FFN, self).__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

残差链接和层归一化
在Transformer中，每个子层（Attention 或 FFN）的输出都不能直接传给下一层，而是要经过add & norm操作:

这里包含两个操作：
残差链接：听名字很高大上，其实很简单，就是把输入和子层的输出直接相加，，随着网络层数加深，梯度可能存在消失或者爆炸的问题，残差提供的是一条 “安全隧道“ ，让梯度可以直接回流浅层。


层归一化
首先我们搞明白，归一化到底是在做什么
不管 LayerNorm 还是 BatchNorm，本质都是同一件事：
```
输出 = (x - 均值) / 标准差
```
把数据拉到均值 0、方差 1 的范围，防止数值越来越大或越来越小，让训练更稳定、收敛更快。
区别在哪：沿哪个方向算均值和标准差

用一个具体例子说明。假设一个 batch 有 3 个句子，每个句子用 4 维向量表示：

```
         特征1  特征2  特征3  特征4
句子1  [  2,     4,     6,     8  ]
句子2  [  1,     3,     5,     7  ]
句子3  [  3,     5,     7,     9  ]
```


BatchNorm（纵向算）：对每个特征，跨所有句子算均值和标准差。
```
特征1 的均值 = (2+1+3)/3 = 2
特征2 的均值 = (4+3+5)/3 = 4
...
↓ 方向：竖着看每一列
```

LayerNorm（横向算）：对每个样本自身，跨所有特征算均值和标准差。
```
句子1 的均值 = (2+4+6+8)/4 = 5
句子2 的均值 = (1+3+5+7)/4 = 4
...
↓ 方向：横着看每一行
```

画个图比较清晰。
```
数据矩阵 (batch=3, features=4):

         特征1  特征2  特征3  特征4
句子1  [  -    -    -    -  ]  ← LayerNorm: 沿这个方向算
句子2  [  -    -    -    -  ]  ← LayerNorm: 沿这个方向算
句子3  [  -    -    -    -  ]  ← LayerNorm: 沿这个方向算
         |      |      |     |
         BatchNorm: 沿这个方向算
```

Norm放在哪里其实也很有讲究：
PostNorm 原始论文： 先做子层计算，再Add & Norm x = Norm(x + SubLayer(x))
PreNorm 后来主流，先Norm，再做子层计算，最后add， x = x + SubLayer(Norm(x))

Pre-Norm 训练更稳定（梯度更好），是现代 Transformer 的主流选择。下面的代码我们先用 Post-Norm（和原始论文一致），方便对照论文理解。

In [13]:

# 先模拟一个 （batch = 2, seq_len = 3, d_model = 4）的输入
x = torch.tensor([[[1.0, 2.0, 3.0, 4.0],
                    [5.0, 6.0, 7.0, 8.0],
                    [9.0, 10., 11., 12.]],
                   [[0.1, 0.2, 0.3, 0.4],
                    [0.5, 0.6, 0.7, 0.8],
                    [0.9, 1.0, 1.1, 1.2]]])

layer_norm = nn.LayerNorm(4)
output = layer_norm(x)
print(f"LayerNorm 输入 第一行: {x[0, 0, :]}")
print(f"归一化前均值：{x[0,0,:].mean().item():.4f}, 标准差：{x[0,0,:].std(unbiased=False).item():.4f}")
print(f"LayerNorm 输出 第一行: {output[0, 0, :]}")
print(f"归一化后均值：{output[0,0,:].mean().item():.4f}, 标准差：{output[0,0,:].std(unbiased=False).item():.4f}")


# 演示下残差链接
sublayer_output = torch.randn(2,3,4) # 模拟子层输出
residual = x + sublayer_output
print(f"输入x的范围： {x.min():.2f} , {x.max():.2f}")
print(f"子层输出的范围： {sublayer_output.min():.2f} , {sublayer_output.max():.2f}")
print(f"残差链接后的范围： {residual.min():.2f} , {residual.max():.2f}")


LayerNorm 输入 第一行: tensor([1., 2., 3., 4.])
归一化前均值：2.5000, 标准差：1.1180
LayerNorm 输出 第一行: tensor([-1.3416, -0.4472,  0.4472,  1.3416], grad_fn=<SliceBackward0>)
归一化后均值：0.0000, 标准差：1.0000
输入x的范围： 0.10 , 12.00
子层输出的范围： -2.48 , 3.22
残差链接后的范围： -1.68 , 13.38


有了Attention， FFN， LayerNorm, 残差链接， 我们就能搭建Encoder的一层积木了， Encoder = SelfAttention + Add & Norm+ FFN + Add & Norm

In [14]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout = 0.1):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, num_heads)
        self.ffn = FFN(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self,x,mask):
        attn_output, _ = self.self_attention(x,x,x,mask)
        x = self.norm1(x + self.dropout(attn_output))

        ffn_output = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_output))
        return x

组装Decoder Layer， Decoder稍微复杂一点，有三个子层，Masked Self Attention, Cross Attention, FFN.
Masked Self Attention 是 Decoder 层的第一个子层，它的输入是上一层的输出，输出是 Decoder 层的输入，掩码的作用是防止看到未来的信息。
Cross Attention 是 Decoder 层的第二个子层，其中输入的Q是 Decoder的， K和V是来自Encoder的。
FFN和之前一样，都是前馈网络，用于增加模型的非线性能力。

虽然上面讲了Mask是为了防止模型看到未来的信息，但是具体是如何实现的呢？
自注意力用的是 tgt_mask, 是一个上三角矩阵，用来屏蔽未来的信息。
因为自注意力的QKV全部来自于Decoder的输入，Decoder输入的是target，也就是答案，我们要玩的是词语接龙，如果你能看到答案，你就不需要预测，也就不会训练预测的能力，就像小朋友如果不撕掉寒暑假作业的答案的话，那就不会认真做作业，直接抄答案就好了，作业的作用就没了。（自律的小朋友除外）
tgt_mask 长这样（下三角矩阵，1 = 能看，0 = 遮蔽）：
```
Q 看 K →   <sos>   I    eat   apple
  <sos>  [  1      0     0      0  ]
  I      [  1      1     0      0  ]
  eat    [  1      1     1      0  ]
  apple  [  1      1     1      1  ]

```

应用mask 前的attention score:
```
            <sos>   I     eat   apple
  <sos>  [  2.1    0.8   -0.3   1.5  ]
  I      [  0.5    3.2    1.1   0.7  ]
  eat    [  0.3    1.0    2.8   0.4  ]
  apple  [  0.1    0.6    0.9   3.0  ]
```
应用 mask 后（mask==0 的位置填 -1e9）：
```
            <sos>   I      eat     apple
  <sos>  [  2.1   -1e9   -1e9    -1e9  ] ← 只能看自己
  I      [  0.5    3.2   -1e9    -1e9  ] ← 只能看 <sos> 和 I
  eat    [  0.3    1.0    2.8    -1e9  ] ← 看不到 apple
  apple  [  0.1    0.6    0.9     3.0  ] ← 全部能看
```

交叉注意力- 用的是src_mask，只需要遮蔽src中的padding token即可，因为目标句中的每个词都要结合原句中的所有有效词，这里排除padding。
src_mask长这样
```
Q 看 K →     我     吃    苹果   <pad>
  <sos>  [   1      1      1      0   ]
  I      [   1      1      1      0   ]
  eat    [   1      1      1      0   ]
  apple  [   1      1      1      0   ]
```

应用mask前:
```
              我     吃    苹果   <pad>
  <sos>  [  0.5    1.2    0.8    0.3  ]
  I      [  0.9    2.1    0.4    0.1  ]
  eat    [  0.3    1.8    2.5    0.6  ]
  apple  [  1.1    0.7    2.9    0.2  ]
```

应用mask后：
```
              我     吃    苹果   <pad>
  <sos>  [  0.5    1.2    0.8   -1e9  ]
  I      [  0.9    2.1    0.4   -1e9  ]
  eat    [  0.3    1.8    2.5   -1e9  ]
  apple  [  1.1    0.7    2.9   -1e9  ]
```




In [15]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout = 0.1):
        super().__init__()
        self.self_attention = MultiHeadAttention(d_model, num_heads)

        self.cross_attention = MultiHeadAttention(d_model, num_heads)

        self.ffn = FFN(d_model, d_ff, dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask, tgt_mask):
        # 自注意力
        attn_output, _ = self.self_attention(x,x,x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))

        # 交叉注意力
        attn_output, attn_map = self.cross_attention(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout(attn_output))

        # 前馈网络
        ffn_output = self.ffn(x)
        x = self.norm3(x + self.dropout(ffn_output))
        return x, attn_map



我们已经组装好了Transformer的核心部件，Encoder和Decoder Layer。现在来测试一下吧，大家也可以手推一下为什么是这些值，更能加深印象。

In [17]:
d_model = 512
num_heads = 8
d_ff = 2048
dropout = 0.1
batch_size = 2
src_len = 10
tgt_len = 5

enc_layer = EncoderLayer(d_model, num_heads, d_ff, dropout)
src_input = torch.randn(batch_size,src_len,d_model)
# 这里我们不使用mask，所以传入None
enc_output = enc_layer(src_input, mask=None)

print(f"Encoder Layer Input: {src_input.shape}")
print(f"Encoder Layer Output: {enc_output.shape}")

dec_layer = DecoderLayer(d_model, num_heads, d_ff, dropout)
tgt_input = torch.randn(batch_size, tgt_len, d_model)

dec_output, cross_attn_map = dec_layer(tgt_input, enc_output, src_mask=None, tgt_mask=None)
print(f"Decoder Layer Input: {tgt_input.shape}")
print(f"Decoder Layer Output: {dec_output.shape}")
print(f"Decoder Layer Cross Attention Map: {cross_attn_map.shape}")

Encoder Layer Input: torch.Size([2, 10, 512])
Encoder Layer Output: torch.Size([2, 10, 512])
Decoder Layer Input: torch.Size([2, 5, 512])
Decoder Layer Output: torch.Size([2, 5, 512])
Decoder Layer Cross Attention Map: torch.Size([2, 8, 5, 10])


恭喜大家，已经完成了Transformer中最难的数学部分，我们现在有了输入处理， 有了计算单元，只要像叠积木一样，把多个Encoder和Decoder串联起来，就能得到完整的“变形金刚“了。